# wildfire inference notebook

This notebook performs inference only. It never trains, tunes, calibrates, or calls `.fit()`. Every fitted weight and preprocessing parameter is loaded from `champion_model.joblib`, produced by the training notebook.

Kaggle workflow:

1. Run the training notebook and save its output as a Kaggle dataset.
2. Attach that output dataset here.
3. Attach the matching history pack for `replay_last_day` / `next_day`:
   - KNN artifact → `/kaggle/input/datasets/lakshay654/california-wildfire-knn`
   - median artifact → `/kaggle/input/datasets/lakshay654/california-wildfire-median`
4. Choose an inference mode in the setup cell.

`fire_analysis2.csv` is not needed at inference (selected cells are stored in the artifact).

Modes:

- `replay_last_day`: default end-to-end verification using the last day in the attached archive;
- `prepared`: score a file already containing the artifact's 113 or 114 features;
- `next_day`: append one complete daily source grid matching the artifact cell set (full grid or a fire-region subset), rebuild causal features, and score it.

For a KNN-trained artifact, `next_day` input must already contain KNN-imputed values and `s2n_knn_imputed`. For a median-trained artifact, the saved pipeline applies its training medians automatically.

If the artifact was trained with `CELL_SUBSET` (`high_fire` or `high_medium_fire`), history and `next_day` grids are filtered to the same selected cells recorded in `data_contract`.


## 1. Load the trained artifact

Attach the training-output dataset (contains `models/champion_model.joblib`) plus the matching history pack:

| Artifact stage | History path |
|---|---|
| `stage_c_knn` | `/kaggle/input/datasets/lakshay654/california-wildfire-knn` |
| `stage_c` | `/kaggle/input/datasets/lakshay654/california-wildfire-median` |

Leave `MODEL_ARTIFACT_PATH` empty to auto-find a single `champion_model.joblib` under `/kaggle/input`, or set it explicitly.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "champion-wildfire-mpl"))

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn import set_config

set_config(display="diagram")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Edit these values in Kaggle.
# After training, attach the saved training-output dataset + the matching history pack:
#   knn artifact     ↔  /kaggle/input/datasets/lakshay654/california-wildfire-knn
#   median artifact  ↔  /kaggle/input/datasets/lakshay654/california-wildfire-median
# fire_analysis2.csv is NOT needed at inference (cell list is inside the artifact).
MODEL_ARTIFACT_PATH = ""  # e.g. "/kaggle/input/<your-training-output>/models/champion_model.joblib"
HISTORY_DATA_DIRECTORY = "/kaggle/input/datasets/lakshay654/california-wildfire-knn"  # or .../california-wildfire-median
INFERENCE_INPUT_FILE = ""  # Required for prepared or next_day; empty OK for replay_last_day
INFERENCE_INPUT_KIND = "replay_last_day"  # replay_last_day | prepared | next_day
# If MODEL_ARTIFACT_PATH is empty, the notebook auto-finds a single champion_model.joblib under /kaggle/input.

default_output = (
    Path("/kaggle/working/champion_inference_outputs")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "notebook_outputs" / "champion_inference"
)
OUTPUT_DIR = Path(os.environ.get("CHAMPION_INFERENCE_OUTPUT_DIR", str(default_output))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def show(value: Any) -> None:
    try:
        from IPython.display import display
        display(value)
    except ImportError:
        print(value)

def find_model_artifact() -> Path:
    configured = MODEL_ARTIFACT_PATH.strip() or os.environ.get("CHAMPION_MODEL_ARTIFACT", "").strip()
    if configured:
        path = Path(configured).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"Configured model artifact does not exist: {path}")
        return path
    candidates: list[Path] = []
    if Path("/kaggle/input").is_dir():
        candidates.extend(Path("/kaggle/input").rglob("champion_model.joblib"))
    current = Path.cwd().resolve()
    candidates.extend(path for base in (current, *current.parents) for path in (base / "notebook_outputs").rglob("champion_model.joblib") if (base / "notebook_outputs").is_dir())
    candidates = list(dict.fromkeys(path.resolve() for path in candidates if path.is_file()))
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "champion_model.joblib was not found. Attach the training notebook output dataset "
            "or set MODEL_ARTIFACT_PATH."
        )
    raise ValueError("Multiple champion_model.joblib files were found; set MODEL_ARTIFACT_PATH explicitly:\n" + "\n".join(map(str, candidates)))

MODEL_PATH = find_model_artifact()
artifact = joblib.load(MODEL_PATH)
required_artifact_keys = {
    "source_stage", "imputation_method", "classifier_pipeline", "ranker_pipeline",
    "probability_calibrator", "feature_columns", "base_features", "source_columns",
    "classifier_weight", "ranker_weight", "data_contract",
}
missing_artifact_keys = sorted(required_artifact_keys - set(artifact))
if missing_artifact_keys:
    raise ValueError(f"Training artifact is missing keys: {missing_artifact_keys}")

print("=" * 72)
print("CHAMPION INFERENCE CONFIGURATION")
print("=" * 72)
print(f"Model artifact:    {MODEL_PATH}")
print(f"Training stage:    {artifact['source_stage']}")
print(f"Cell subset:       {artifact.get('cell_subset', artifact.get('data_contract', {}).get('cell_subset', 'all'))}")
print(f"Imputation:        {artifact['imputation_method']}")
print(f"Model features:    {len(artifact['feature_columns'])}")
print(f"Input kind:        {INFERENCE_INPUT_KIND}")
print(f"Output directory:  {OUTPUT_DIR}")
print("=" * 72)


## 2. Inspect the exact fitted pipelines

These are loaded objects from training, not newly constructed estimators.

In [ ]:
print("Loaded classifier pipeline")
show(artifact["classifier_pipeline"])
print("Loaded ranker pipeline")
show(artifact["ranker_pipeline"])
show(pd.DataFrame([{
    "artifact": MODEL_PATH.name,
    "source_stage": artifact["source_stage"],
    "cell_subset": artifact.get("cell_subset", artifact.get("data_contract", {}).get("cell_subset", "all")),
    "imputation": artifact["imputation_method"],
    "features": len(artifact["feature_columns"]),
    "expected_grid_cells": artifact.get("data_contract", {}).get("expected_grid_cells"),
    "classifier_weight": artifact["classifier_weight"],
    "ranker_weight": artifact["ranker_weight"],
}]))


## 3. Causal feature engineering for history-based inference

This collapsed cell contains the same deterministic feature formulas used in training. It is not model fitting. Prepared-feature inference bypasses it.

In [ ]:
def rolling_matrix(values: np.ndarray, window: int, operation: str) -> np.ndarray:
    rolling = pd.DataFrame(values.T).rolling(window=window, min_periods=1)
    return getattr(rolling, operation)().to_numpy(dtype="float32").T

def simple_neighbors(frame: pd.DataFrame, cells: int, days: int) -> list[np.ndarray]:
    coordinates = frame.iloc[np.arange(cells) * days][["latitude", "longitude"]].to_numpy(dtype="float64")
    result = []
    for latitude, longitude in coordinates:
        delta_lat = np.abs(coordinates[:, 0] - latitude)
        delta_lon = np.abs(coordinates[:, 1] - longitude)
        mask = (
            (delta_lat <= 0.251)
            & (delta_lon <= 0.251)
            & ~((delta_lat < 1e-9) & (delta_lon < 1e-9))
        )
        result.append(np.flatnonzero(mask))
    return result

def sum_neighbors(values: np.ndarray, neighbors: list[np.ndarray]) -> np.ndarray:
    output = np.zeros_like(values, dtype="float32")
    for index, adjacent in enumerate(neighbors):
        if adjacent.size:
            output[index] = values[adjacent].sum(axis=0)
    return output

def directional_geometry(coordinates: np.ndarray):
    result = []
    for latitude, longitude in coordinates:
        delta_lat = latitude - coordinates[:, 0]
        delta_lon = (longitude - coordinates[:, 1]) * np.cos(np.deg2rad(latitude))
        distance = np.sqrt(delta_lat**2 + delta_lon**2)
        adjacent = np.flatnonzero((distance > 1e-9) & (distance <= 0.36))
        result.append((
            adjacent,
            (delta_lon[adjacent] / distance[adjacent]).astype("float32"),
            (delta_lat[adjacent] / distance[adjacent]).astype("float32"),
            (1.0 / (distance[adjacent] + 0.05)).astype("float32"),
        ))
    return result

def directional_counts(fire: np.ndarray, wind_sin: np.ndarray, wind_cos: np.ndarray, geometry):
    cells, days = fire.shape
    upwind = np.zeros((cells, days), dtype="float32")
    downwind = np.zeros((cells, days), dtype="float32")
    crosswind = np.zeros((cells, days), dtype="float32")
    distance_weighted = np.zeros((cells, days), dtype="float32")
    wind_east, wind_north = -wind_sin, -wind_cos
    for index, (adjacent, east, north, inverse_distance) in enumerate(geometry):
        if not adjacent.size:
            continue
        neighbor_fire = fire[adjacent]
        alignment = wind_east[index][None, :] * east[:, None] + wind_north[index][None, :] * north[:, None]
        upwind[index] = (neighbor_fire * np.maximum(alignment, 0) * inverse_distance[:, None]).sum(axis=0)
        downwind[index] = (neighbor_fire * np.maximum(-alignment, 0) * inverse_distance[:, None]).sum(axis=0)
        crosswind[index] = (
            neighbor_fire * np.sqrt(np.maximum(1.0 - alignment**2, 0)) * inverse_distance[:, None]
        ).sum(axis=0)
        distance_weighted[index] = (neighbor_fire * inverse_distance[:, None]).sum(axis=0)
    return upwind, downwind, crosswind, distance_weighted

def build_features(frame: pd.DataFrame, raw_base_features: list[str]):
    cells = int(frame["cell_id"].nunique())
    days = int(frame["label_date"].nunique())

    # Calendar cycles.
    day_of_year = frame["eo_asof_date"].dt.dayofyear.to_numpy(dtype="float32")
    month = frame["eo_asof_date"].dt.month.to_numpy(dtype="float32")
    calendar = pd.DataFrame({
        "day_of_year_sin": np.sin(2 * np.pi * day_of_year / 365.25),
        "day_of_year_cos": np.cos(2 * np.pi * day_of_year / 365.25),
        "month_sin": np.sin(2 * np.pi * month / 12),
        "month_cos": np.cos(2 * np.pi * month / 12),
    }, index=frame.index).astype("float32")
    frame = pd.concat([frame, calendar], axis=1)

    # Weather physics and history ending at D-5.
    temperature_c = frame["t2m_mean"].to_numpy(dtype="float64") - 273.15
    dewpoint_c = frame["d2m_mean"].to_numpy(dtype="float64") - 273.15
    saturation = 0.6108 * np.exp(17.27 * temperature_c / np.maximum(temperature_c + 237.3, 1e-6))
    actual = 0.6108 * np.exp(17.27 * dewpoint_c / np.maximum(dewpoint_c + 237.3, 1e-6))
    vpd = np.maximum(saturation - actual, 0).astype("float32")
    weather = {
        "vpd_kpa": vpd,
        "vpd_wind_interaction": vpd * frame["wind_speed_mean"].to_numpy(dtype="float32"),
        "vpd_soil_deficit_interaction": vpd * (1 - np.clip(frame["soil_moisture_index"], 0, 1)),
        "heat_soil_deficit_interaction": (
            np.maximum(frame["t2m_max"].to_numpy(dtype="float32") - 273.15, 0)
            * (1 - np.clip(frame["swvl1_mean"], 0, 1))
        ),
        "wind_gust_ratio": frame["i10fg_max"].to_numpy(dtype="float32")
        / (frame["wind_speed_mean"].to_numpy(dtype="float32") + 0.1),
    }
    rolling_specs = {
        "t2m_max": ("max",), "rh_mean": ("min",), "tp_sum_mm": ("sum",),
        "wind_speed_mean": ("max",), "i10fg_max": ("max",),
        "swvl1_mean": ("mean",), "vpd_kpa": ("max", "mean"),
    }
    arrays = {
        name: (weather[name] if name in weather else frame[name].to_numpy(dtype="float32")).reshape(cells, days)
        for name in rolling_specs
    }
    for name, operations in rolling_specs.items():
        for window in (14, 30):
            for operation in operations:
                weather[f"{name}_{operation}_{window}d"] = rolling_matrix(
                    arrays[name], window, operation
                ).reshape(-1)
    temperature = frame["t2m_max"].to_numpy(dtype="float32").reshape(cells, days)
    soil = frame["swvl1_mean"].to_numpy(dtype="float32").reshape(cells, days)
    weather["t2m_max_anomaly_30d"] = (temperature - rolling_matrix(temperature, 30, "mean")).reshape(-1)
    weather["swvl1_anomaly_30d"] = (soil - rolling_matrix(soil, 30, "mean")).reshape(-1)
    weather_frame = pd.DataFrame(weather, index=frame.index).astype("float32")
    frame = pd.concat([frame, weather_frame], axis=1)

    # Fire history ending at D-1 (two target rows behind label day D+1).
    target = frame["y_fire"].to_numpy(dtype="float32").reshape(cells, days)
    lag2 = np.zeros_like(target, dtype="float32")
    lag2[:, 2:] = target[:, :-2]
    history7 = rolling_matrix(lag2, 7, "sum")
    history30 = rolling_matrix(lag2, 30, "sum")
    neighbors = simple_neighbors(frame, cells, days)
    neighbor_lag2 = sum_neighbors(lag2, neighbors)
    neighbor_7d = sum_neighbors(history7, neighbors)
    last_positive = np.full(cells, -10_000, dtype="int32")
    days_since = np.full_like(target, 365, dtype="float32")
    for day in range(days):
        positive = lag2[:, day] > 0
        last_positive[positive] = day
        seen = last_positive > -10_000
        days_since[seen, day] = np.minimum(day - last_positive[seen], 365)
    observed = np.maximum(np.arange(days, dtype="float32") - 1, 0)
    expanding_rate = (np.cumsum(lag2, axis=1, dtype="float32") + 1.0) / (observed[None, :] + 100.0)
    fire = {
        "fire_cell_lag2": lag2.reshape(-1),
        "fire_cell_count_7d_lag2": history7.reshape(-1),
        "fire_cell_count_30d_lag2": history30.reshape(-1),
        "fire_cell_any_7d_lag2": (history7 > 0).astype("float32").reshape(-1),
        "fire_cell_days_since_lag2": days_since.reshape(-1),
        "fire_cell_expanding_rate_lag2": expanding_rate.reshape(-1),
        "fire_neighbor_count_lag2": neighbor_lag2.reshape(-1),
        "fire_neighbor_count_7d_lag2": neighbor_7d.reshape(-1),
        "fire_neighbor_any_7d_lag2": (neighbor_7d > 0).astype("float32").reshape(-1),
        "fire_statewide_cells_7d_lag2": np.tile(history7.sum(axis=0), cells).astype("float32"),
    }
    fire_frame = pd.DataFrame(fire, index=frame.index).astype("float32")
    frame = pd.concat([frame, fire_frame], axis=1)

    # Wind-aware neighboring fire context.
    coordinates = frame.iloc[np.arange(cells) * days][["latitude", "longitude"]].to_numpy(dtype="float64")
    geometry = directional_geometry(coordinates)
    wind_sin = frame["wind_dir_sin"].to_numpy(dtype="float32").reshape(cells, days)
    wind_cos = frame["wind_dir_cos"].to_numpy(dtype="float32").reshape(cells, days)
    upwind_lag2, _, _, distance_lag2 = directional_counts(lag2, wind_sin, wind_cos, geometry)
    upwind_7d, downwind_7d, crosswind_7d, distance_7d = directional_counts(
        history7, wind_sin, wind_cos, geometry
    )
    wind = frame["wind_speed_mean"].to_numpy(dtype="float32")
    vpd = frame["vpd_kpa"].to_numpy(dtype="float32")
    soil_deficit = 1 - np.clip(frame["soil_moisture_index"].to_numpy(dtype="float32"), 0, 1)
    vegetation = np.clip(
        frame["cvh_mean"].to_numpy(dtype="float32") + frame["cvl_mean"].to_numpy(dtype="float32"), 0, 1
    )
    recent_context = np.maximum(
        frame["fire_cell_any_7d_lag2"].to_numpy(dtype="float32"),
        frame["fire_neighbor_any_7d_lag2"].to_numpy(dtype="float32"),
    )
    directional = {
        "fire_upwind_count_lag2": upwind_lag2.reshape(-1),
        "fire_upwind_count_7d_lag2": upwind_7d.reshape(-1),
        "fire_downwind_count_7d_lag2": downwind_7d.reshape(-1),
        "fire_crosswind_count_7d_lag2": crosswind_7d.reshape(-1),
        "fire_distance_weighted_count_lag2": distance_lag2.reshape(-1),
        "fire_distance_weighted_count_7d_lag2": distance_7d.reshape(-1),
        "fire_wind_spread_potential_lag2": upwind_lag2.reshape(-1) * wind,
        "fire_wind_spread_potential_7d_lag2": upwind_7d.reshape(-1) * wind,
        "fire_context_vpd_interaction": frame["fire_neighbor_count_7d_lag2"].to_numpy(dtype="float32") * vpd,
        "fire_context_dry_windy_interaction": recent_context * vpd * wind * soil_deficit,
        "ignition_dry_windy_index": vpd * wind * soil_deficit,
        "fuel_dryness_index": vpd * soil_deficit * vegetation,
        "vpd_short_long_trend": frame["vpd_kpa_mean_14d"] - frame["vpd_kpa_mean_30d"],
        "recent_fire_context": recent_context,
    }
    directional_frame = pd.DataFrame(directional, index=frame.index).astype("float32")
    frame = pd.concat([frame, directional_frame], axis=1)

    # The locked contract excludes a redundant source-availability flag.
    selected_base = [name for name in raw_base_features if name != "s5n_available"]
    feature_columns = list(dict.fromkeys([
        *selected_base,
        "latitude", "longitude",
        *calendar.columns,
        *weather_frame.columns,
        *fire_frame.columns,
        *directional_frame.columns,
    ]))
    groups = {
        "source_after_constant_removal": len(raw_base_features),
        "source_used_by_model": len(selected_base),
        "geographic": 2,
        "calendar": len(calendar.columns),
        "weather_and_interactions": len(weather_frame.columns),
        "causal_fire_history": len(fire_frame.columns),
        "wind_and_context": len(directional_frame.columns),
        "total": len(feature_columns),
    }
    return frame, feature_columns, groups

## 4. Load history, prepare inputs, and score with the trained artifact

Helpers validate stage compatibility against the attached `california-wildfire-knn` / `california-wildfire-median` pack, feature order, grid completeness (full archive or fire-region subset from the artifact), timing offsets, and next-day continuity before prediction.


In [ ]:
def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

def read_table(path: str | Path) -> pd.DataFrame:
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Input file does not exist: {path}")
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError("Input must be Parquet, CSV, JSON, JSONL, or NDJSON")

def history_candidate_roots() -> list[Path]:
    candidates: list[Path] = []
    configured = HISTORY_DATA_DIRECTORY.strip() or os.environ.get("CHAMPION_HISTORY_DATA_DIR", "").strip()
    if configured:
        candidates.append(Path(configured).expanduser())
    if Path("/kaggle/input").is_dir():
        candidates.extend(path.parent for path in Path("/kaggle/input").rglob("dataset_metadata.json"))
    current = Path.cwd().resolve()
    candidates.extend(base / "Archive" for base in (current, *current.parents))
    return list(dict.fromkeys(path.resolve() for path in candidates))

def locate_history(expected_stage: str) -> dict[str, Path]:
    checked: list[str] = []
    for root in history_candidate_roots():
        folder = "stage_c_knn" if expected_stage == "stage_c_knn" else "stage_c"
        layouts = [
            {
                "root": root, "all": root / "all.parquet", "meta": root / "meta.json",
                "dataset_metadata": root / "dataset_metadata.json", "features": root / "feature_columns.json",
            },
            {
                "root": root / folder, "all": root / folder / "all.parquet", "meta": root / folder / "meta.json",
                "dataset_metadata": root / folder / "metadata" / "dataset_metadata.json",
                "features": root / folder / "metadata" / "feature_columns.json",
            },
            {
                "root": root, "all": root / "stage_c" / "all.parquet", "meta": root / "meta.json",
                "dataset_metadata": root / "stage_c" / "metadata" / "dataset_metadata.json",
                "features": root / "stage_c" / "metadata" / "feature_columns.json",
            },
        ]
        for layout in layouts:
            ready = all(path.is_file() for key, path in layout.items() if key != "root")
            stage = read_json(layout["dataset_metadata"]).get("stage") if ready else None
            checked.append(f"{layout['root']} (stage={stage})")
            if ready and stage == expected_stage:
                return layout
    raise FileNotFoundError(
        f"No four-file history dataset with stage={expected_stage} was found. "
        "Set HISTORY_DATA_DIRECTORY. Checked:\n" + "\n".join(f"  - {item}" for item in checked)
    )

def clean_source_table(frame: pd.DataFrame, model_artifact: dict):
    frame = frame.copy()
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        frame[name] = pd.to_datetime(frame[name]).dt.normalize()
    frame = frame.sort_values(["cell_id", "label_date"]).reset_index(drop=True)
    base = model_artifact["base_features"]
    cleanup: dict[str, int] = {}

    if model_artifact["source_stage"] == "stage_c_knn":
        frame[base] = frame[base].apply(pd.to_numeric, errors="raise").astype("float32")
        flag = model_artifact["missing_flag_column"]
        flag_values = set(frame[flag].dropna().unique().tolist())
        if not flag_values.issubset({0.0, 1.0}):
            raise ValueError(f"{flag} must be binary")
        if not np.isfinite(frame[base].to_numpy(dtype="float32", copy=False)).all():
            raise ValueError("KNN source data contains NaN or infinity")
        flagged = frame[flag].eq(1)
        if not frame.loc[flagged, "s2n_available"].eq(0).all():
            raise ValueError("KNN-imputed rows must retain s2n_available=0")
        cleanup["knn_imputed_rows_flagged"] = int(flagged.sum())
    else:
        s2_invalid = frame["s2n_available"].ne(1)
        s2_values = [name for name in base if name.startswith("s2n_") and name != "s2n_available"]
        frame.loc[s2_invalid, s2_values] = np.nan
        frame.loc[s2_invalid, "s2n_available"] = 0.0
        cleanup["sentinel2_rows_marked_missing"] = int(s2_invalid.sum())
        s5_invalid = frame["s5n_available"].ne(1)
        s5_values = [name for name in base if name.startswith("s5n_") and name != "s5n_available"]
        frame.loc[s5_invalid, s5_values] = 0.0
        frame.loc[s5_invalid, "s5n_available"] = 0.0
        cleanup["sentinel5p_rows_zeroed"] = int(s5_invalid.sum())
        frame[base] = frame[base].astype("float32")

    for name in ("swvl1_mean", "swvl2_mean", "soil_moisture_index", "swvl1_mean_7d"):
        cleanup[f"{name}_negative_rows_clipped"] = int(frame[name].lt(0).sum())
        frame[name] = frame[name].clip(lower=0)
    frame["year"] = frame["label_date"].dt.year.astype("int16")
    return frame, cleanup

def validate_complete_grid(frame: pd.DataFrame) -> None:
    cells = int(frame["cell_id"].nunique())
    days = int(frame["label_date"].nunique())
    if len(frame) != cells * days or frame.duplicated(["cell_id", "label_date"]).any():
        raise ValueError("Source data must form a complete cell-by-day grid")
    if not (frame["label_date"] - frame["eo_asof_date"]).dt.days.eq(1).all():
        raise ValueError("label_date must equal eo_asof_date + 1 day")
    if not (frame["eo_asof_date"] - frame["feature_end_date"]).dt.days.eq(5).all():
        raise ValueError("eo_asof_date must equal feature_end_date + 5 days")


def artifact_cell_subset(model_artifact: dict) -> str:
    return str(
        model_artifact.get("cell_subset")
        or model_artifact.get("data_contract", {}).get("cell_subset")
        or "all"
    )


def artifact_selected_cells(model_artifact: dict) -> list | None:
    contract = model_artifact.get("data_contract", {}) or {}
    cells = contract.get("selected_cell_ids")
    if cells:
        return [str(cell) for cell in cells]
    if artifact_cell_subset(model_artifact) == "all":
        return None
    raise ValueError(
        "Artifact cell_subset is not 'all' but data_contract.selected_cell_ids is missing. "
        "Retrain with the updated training notebook."
    )


def filter_frame_to_artifact_cells(frame: pd.DataFrame, model_artifact: dict) -> pd.DataFrame:
    selected = artifact_selected_cells(model_artifact)
    if selected is None:
        return frame
    selected_set = set(selected)
    filtered = frame.loc[frame["cell_id"].astype(str).isin(selected_set)].copy()
    if filtered.empty:
        raise ValueError("No history rows remain after applying the artifact cell subset")
    missing = sorted(selected_set - set(filtered["cell_id"].astype(str).unique()))
    if missing:
        raise ValueError(
            f"History is missing {len(missing)} artifact cells after subset filter "
            f"(example: {missing[:5]})"
        )
    return filtered


def build_history_features(history_path: Path, model_artifact: dict):
    history = pd.read_parquet(history_path, columns=model_artifact["source_columns"])
    history = filter_frame_to_artifact_cells(history, model_artifact)
    history, cleanup = clean_source_table(history, model_artifact)
    validate_complete_grid(history)
    cleanup = dict(cleanup)
    cleanup["cell_subset"] = artifact_cell_subset(model_artifact)
    cleanup["cells_after_subset"] = int(history["cell_id"].nunique())
    engineered, generated_features, _ = build_features(history, model_artifact["base_features"])
    if generated_features != model_artifact["feature_columns"]:
        raise ValueError("Generated feature order differs from the training artifact")
    return engineered, cleanup

def prepare_next_day(
    new_daily_grid: pd.DataFrame,
    history_path: Path,
    model_artifact: dict,
) -> tuple[pd.DataFrame, dict]:
    source_columns = model_artifact["source_columns"]
    required_without_target = [name for name in source_columns if name != "y_fire"]
    missing = sorted(set(required_without_target) - set(new_daily_grid.columns))
    if missing:
        raise ValueError(f"Next-day input is missing {len(missing)} columns: {missing[:15]}")
    custom = new_daily_grid[required_without_target].copy()
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        custom[name] = pd.to_datetime(custom[name]).dt.normalize()
    if custom["label_date"].nunique() != 1 or custom.duplicated(["cell_id", "label_date"]).any():
        raise ValueError("next_day input must contain one unique, duplicate-free label_date")

    history = pd.read_parquet(history_path, columns=source_columns)
    history = filter_frame_to_artifact_cells(history, model_artifact)
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        history[name] = pd.to_datetime(history[name]).dt.normalize()
    forecast_date = custom["label_date"].iloc[0]
    expected_date = history["label_date"].max() + pd.Timedelta(days=1)
    if forecast_date != expected_date:
        raise ValueError(f"next_day label_date must be {expected_date.date()}; received {forecast_date.date()}")
    expected_cells = set(history["cell_id"].astype(str).unique())
    custom_cells = set(custom["cell_id"].astype(str).unique())
    if custom_cells != expected_cells:
        raise ValueError(
            "next_day must contain exactly the artifact cell set "
            f"(expected {len(expected_cells)} cells for subset "
            f"{artifact_cell_subset(model_artifact)})"
        )
    archive_grid = history.sort_values("label_date").drop_duplicates("cell_id", keep="last")[["cell_id", "latitude", "longitude"]]
    coordinates = custom[["cell_id", "latitude", "longitude"]].merge(
        archive_grid, on="cell_id", how="left", suffixes=("_new", "_archive"), validate="one_to_one"
    )
    if not (
        np.allclose(coordinates["latitude_new"], coordinates["latitude_archive"], atol=1e-6)
        and np.allclose(coordinates["longitude_new"], coordinates["longitude_archive"], atol=1e-6)
    ):
        raise ValueError("next_day coordinates do not match the training grid")

    custom["y_fire"] = np.int8(0)
    combined = pd.concat([history, custom[source_columns]], ignore_index=True)
    combined, cleanup = clean_source_table(combined, model_artifact)
    validate_complete_grid(combined)
    engineered, generated_features, _ = build_features(combined, model_artifact["base_features"])
    if generated_features != model_artifact["feature_columns"]:
        raise ValueError("Generated feature order differs from the training artifact")
    prepared = engineered.loc[engineered["label_date"].eq(forecast_date)].drop(columns="y_fire").copy()
    return prepared, cleanup

def within_day_percentile(score: np.ndarray, dates: pd.Series) -> np.ndarray:
    table = pd.DataFrame({"date": pd.to_datetime(dates).to_numpy(), "score": score, "position": np.arange(len(score))})
    table["percentile"] = table.groupby("date", sort=False)["score"].rank(method="average", pct=True)
    return table.sort_values("position")["percentile"].to_numpy(dtype=float)

def score_prepared(prepared: pd.DataFrame, model_artifact: dict) -> pd.DataFrame:
    features = model_artifact["feature_columns"]
    missing = sorted({"label_date", *features} - set(prepared.columns))
    if missing:
        raise ValueError(f"Prepared input is missing {len(missing)} columns: {missing[:15]}")
    table = prepared.copy()
    table["label_date"] = pd.to_datetime(table["label_date"]).dt.normalize()
    table[features] = table[features].apply(pd.to_numeric, errors="raise")
    if table.empty:
        raise ValueError("Inference input is empty")

    raw_probability = model_artifact["classifier_pipeline"].predict_proba(table[features])[:, 1]
    clipped = np.clip(raw_probability, 1e-7, 1 - 1e-7)
    calibrated = model_artifact["probability_calibrator"].predict_proba(
        np.log(clipped / (1 - clipped)).reshape(-1, 1)
    )[:, 1]
    sort_columns = ["label_date"] + (["cell_id"] if "cell_id" in table.columns else [])
    rank_input = table.sort_values(sort_columns)
    rank_sorted = model_artifact["ranker_pipeline"].predict(rank_input[features])
    rank_score = pd.Series(rank_sorted, index=rank_input.index).reindex(table.index).to_numpy(dtype=float)
    alert_score = (
        model_artifact["classifier_weight"] * within_day_percentile(raw_probability, table["label_date"])
        + model_artifact["ranker_weight"] * within_day_percentile(rank_score, table["label_date"])
    )
    identifiers = [name for name in (
        "feature_end_date", "eo_asof_date", "label_date", "cell_id",
        "latitude", "longitude", "y_fire",
    ) if name in table.columns]
    result = table[identifiers].copy()
    result["p_fire_raw"] = raw_probability.astype("float32")
    result["p_fire"] = calibrated.astype("float32")
    result["rank_score"] = rank_score.astype("float32")
    result["alert_score"] = alert_score.astype("float32")
    result["daily_priority_rank"] = result.groupby("label_date")["alert_score"].rank(method="first", ascending=False).astype("int32")
    result["alert_top_25"] = result["daily_priority_rank"].le(25)
    return result.sort_values(["label_date", "daily_priority_rank"]).reset_index(drop=True)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## 5. Generate and save inference predictions

Default mode `replay_last_day` scores the last day in the attached history pack. For `prepared` / `next_day`, set `INFERENCE_INPUT_FILE` to your uploaded Parquet under `/kaggle/input/...`.


In [ ]:
input_kind = os.environ.get("CHAMPION_INFERENCE_KIND", INFERENCE_INPUT_KIND).strip().lower()
input_path_value = INFERENCE_INPUT_FILE.strip() or os.environ.get("CHAMPION_INFERENCE_FILE", "").strip()
cleanup_summary: dict[str, int] = {}
history_info = None

if input_kind == "prepared":
    if not input_path_value:
        raise ValueError("INFERENCE_INPUT_FILE is required for prepared inference")
    prepared_features = read_table(input_path_value)
    inference_source = str(Path(input_path_value))
elif input_kind in {"replay_last_day", "next_day"}:
    history_info = locate_history(artifact["source_stage"])
    history_path = history_info["all"]
    if input_kind == "replay_last_day":
        engineered_history, cleanup_summary = build_history_features(history_path, artifact)
        replay_date = engineered_history["label_date"].max()
        prepared_features = engineered_history.loc[engineered_history["label_date"].eq(replay_date)].copy()
        inference_source = f"archive replay for {replay_date.date()}"
        del engineered_history
        gc.collect()
    else:
        if not input_path_value:
            raise ValueError("INFERENCE_INPUT_FILE is required for next_day inference")
        custom_grid = read_table(input_path_value)
        prepared_features, cleanup_summary = prepare_next_day(custom_grid, history_path, artifact)
        inference_source = str(Path(input_path_value))
else:
    raise ValueError("INFERENCE_INPUT_KIND must be replay_last_day, prepared, or next_day")

predictions = score_prepared(prepared_features, artifact)
predictions_path = OUTPUT_DIR / "inference_predictions.parquet"
predictions.to_parquet(predictions_path, index=False)
summary_path = OUTPUT_DIR / "inference_summary.json"
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_artifact": str(MODEL_PATH),
    "model_sha256": sha256(MODEL_PATH),
    "source_stage": artifact["source_stage"],
    "cell_subset": artifact.get("cell_subset", artifact.get("data_contract", {}).get("cell_subset", "all")),
    "expected_grid_cells": artifact.get("data_contract", {}).get("expected_grid_cells"),
    "imputation_method": artifact["imputation_method"],
    "input_kind": input_kind,
    "input_source": inference_source,
    "rows": len(predictions),
    "cells": int(predictions["cell_id"].nunique()) if "cell_id" in predictions.columns else None,
    "label_dates": int(predictions["label_date"].nunique()),
    "cleanup": cleanup_summary,
    "software": {
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__, "lightgbm": lgb.__version__,
    },
}
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(f"Loaded training artifact: {MODEL_PATH}")
print(f"Inference source:         {inference_source}")
print(f"Scored rows:              {len(predictions):,}")
print(f"Label dates:              {predictions['label_date'].nunique():,}")
print(f"Predictions:              {predictions_path}")
print(f"Summary:                  {summary_path}")
show(predictions.head(25))


## 6. Outputs

Written under `/kaggle/working/champion_inference_outputs/` (or the local notebook output folder):

- `inference_predictions.parquet`: calibrated probability, rank score, blended alert score, and daily priority rank.
- `inference_summary.json`: model checksum, source stage, cell subset, input mode, row count, and software versions.

The model artifact checksum connects every prediction file back to the exact training output used. History stage must match the artifact (`california-wildfire-knn` ↔ KNN, `california-wildfire-median` ↔ median).
